# F1 Podium Prediction — Interpretable ML Project

**Goal:** Predict whether a driver will finish on the podium (top 3) using historical Ergast F1 data.

**ML concepts covered:**
- Data loading, merging, and cleaning
- Feature engineering
- Binary classification (Decision Tree, Logistic Regression, Random Forest)
- Model evaluation (accuracy, F1-score, confusion matrix, ROC/AUC)
- Interpretability (tree visualisation, feature importance, error analysis)

---
## 0. Install & Import Libraries

We use standard scientific Python libraries:
- **pandas** — data loading, merging, feature engineering
- **numpy** — numerical operations
- **scikit-learn** — ML models and evaluation
- **matplotlib / seaborn** — visualisation

In [ ]:
# Uncomment if running for the first time
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Make all plots look clean
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')
print('Libraries loaded successfully.')

: 

---
## 1. Load the Ergast Data

Download the Ergast CSV bundle from:
https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020

Place all CSVs in a folder called `f1_data/` next to this notebook.

We load 4 tables:
- `results.csv` — every driver's race result (position, points, status)
- `qualifying.csv` — qualifying times and grid positions
- `races.csv` — race metadata (year, circuit, name)
- `constructors.csv` — team names

In [ ]:
DATA_DIR = 'f1_data/'  # Change this path if needed

results      = pd.read_csv(DATA_DIR + 'results.csv')
qualifying   = pd.read_csv(DATA_DIR + 'qualifying.csv')
races        = pd.read_csv(DATA_DIR + 'races.csv')
constructors = pd.read_csv(DATA_DIR + 'constructors.csv')
drivers      = pd.read_csv(DATA_DIR + 'drivers.csv')

# Quick sanity check — how many rows does each table have?
for name, df in [('results', results), ('qualifying', qualifying),
                 ('races', races), ('constructors', constructors), ('drivers', drivers)]:
    print(f'{name:20s}: {len(df):,} rows, {df.shape[1]} columns')

---
## 2. Merge & Filter

We combine all tables into one flat DataFrame.

The join key is `raceId` (links results → races) and `driverId` (links everything to the driver).

We then filter to **2010–2023** — the modern hybrid/V8 era. Using the full 1950–2023 dataset would mix very different regulatory eras and hurt model quality.

In [ ]:
# Step 1: Merge results with race metadata (year, circuit)
df = results.merge(races[['raceId', 'year', 'circuitId', 'name']], on='raceId', how='left')

# Step 2: Add constructor (team) name
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left',
              suffixes=('_race', '_constructor'))

# Step 3: Add driver nationality (for home race feature later)
df = df.merge(drivers[['driverId', 'nationality', 'surname']], on='driverId', how='left')

# Step 4: Merge qualifying grid position
# We only need position (grid start) from qualifying
qual_cols = qualifying[['raceId', 'driverId', 'position']].rename(columns={'position': 'qual_position'})
df = df.merge(qual_cols, on=['raceId', 'driverId'], how='left')

# Step 5: Filter to modern era
df = df[(df['year'] >= 2010) & (df['year'] <= 2023)].copy()

# Step 6: Replace '\\N' (Ergast's NULL marker) with NaN
df.replace('\\N', np.nan, inplace=True)

# Step 7: Convert numeric columns that may have been loaded as strings
for col in ['positionOrder', 'points', 'grid', 'qual_position', 'laps']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Dataset shape after merge & filter: {df.shape}')
df[['year', 'name_race', 'surname', 'name_constructor', 'grid', 'positionOrder', 'points']].head(10)

---
## 3. Create the Target Variable

Our target is binary:
- **1 = podium** — driver finished in position 1, 2, or 3
- **0 = no podium** — driver finished 4th or lower (or DNF)

We also drop rows where the driver **did not finish (DNF)**, because `positionOrder` for a DNF reflects classification order, not true performance. You could also keep them — an interesting experiment!

> **Class imbalance note:** In any race, only 3 of 20 drivers podium. That's ~15% positive rate. This is real-world imbalance and we'll discuss its effect on metrics.

In [ ]:
# Create the binary target
df['podium'] = (df['positionOrder'] <= 3).astype(int)

# Remove rows where driver retired / was classified but didn't really race
# statusId 1 = 'Finished', small statusIds generally mean race completion
# Simpler approach: keep rows where positionOrder is valid (1-20)
df = df[df['positionOrder'].between(1, 20)].copy()

# Check class balance
balance = df['podium'].value_counts(normalize=True)
print('Class balance:')
print(f'  No podium (0): {balance[0]:.1%}')
print(f'  Podium    (1): {balance[1]:.1%}')
print(f'\nTotal rows: {len(df):,}')

# Visualise class balance
fig, ax = plt.subplots(figsize=(4, 3))
df['podium'].value_counts().plot(kind='bar', ax=ax, color=['#5B8DB8', '#E07B54'], edgecolor='white')
ax.set_xticklabels(['No podium', 'Podium'], rotation=0)
ax.set_title('Target class distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

---
## 4. Exploratory Data Analysis (EDA)

Before modelling, always look at your data. We want to understand:
1. How strongly does grid position predict podium?
2. Which constructors have the highest podium rates?

These plots also foreshadow what features will matter most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Plot 1: Grid position vs podium rate ---
# Group by grid position, compute what % of time that position led to a podium
grid_podium = (
    df[df['grid'] <= 20]
    .groupby('grid')['podium']
    .mean()
    .reset_index()
)
axes[0].bar(grid_podium['grid'], grid_podium['podium'] * 100, color='#5B8DB8', edgecolor='white')
axes[0].set_xlabel('Qualifying grid position')
axes[0].set_ylabel('Podium rate (%)')
axes[0].set_title('Grid position → podium rate')
axes[0].axvline(x=3.5, color='red', linestyle='--', alpha=0.5, label='Top 3 grid')
axes[0].legend()

# --- Plot 2: Top 10 constructors by podium rate ---
constructor_stats = (
    df.groupby('name_constructor')['podium']
    .agg(['mean', 'count'])
    .reset_index()
)
# Only include constructors with at least 50 race entries
constructor_stats = constructor_stats[constructor_stats['count'] >= 50]
constructor_stats = constructor_stats.nlargest(10, 'mean')

axes[1].barh(constructor_stats['name_constructor'], constructor_stats['mean'] * 100,
             color='#E07B54', edgecolor='white')
axes[1].set_xlabel('Podium rate (%)')
axes[1].set_title('Top 10 constructors by podium rate (2010–2023)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 5. Feature Engineering

Raw data rarely feeds straight into a model. We need to transform columns into **informative numeric features**.

| Feature | What it captures | Why it helps |
|---|---|---|
| `grid_position` | Qualifying result | Best single predictor of race outcome |
| `rolling_pts_3` | Driver form last 3 races | Recent momentum — is the driver in form? |
| `constructor_avg_pos` | Team's season avg finish | Car quality proxy |
| `is_top3_grid` | Binary: started P1–P3 | Captures the 'pole advantage' non-linearity |
| `year_norm` | Normalised year | Proxy for rule-era and car development |

> **Key insight:** Rolling features require sorting by date first, and computing them *before* the current race (no data leakage!).

In [ ]:
# Sort chronologically — essential before creating any rolling/lag features
df = df.sort_values(['year', 'raceId', 'driverId']).reset_index(drop=True)

# ── Feature 1: Grid position (already exists) ──────────────────────────────
# Fill missing qualifying data with grid column (some races have no separate qualifying)
df['grid_position'] = df['qual_position'].fillna(df['grid'])

# ── Feature 2: Rolling points over last 3 races (per driver) ───────────────
# shift(1) is critical — it means 'points from the race BEFORE this one'
# Without shift(), we'd be using the current race's result to predict itself = data leakage!
df['rolling_pts_3'] = (
    df.groupby('driverId')['points']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# ── Feature 3: Constructor average finishing position this season ───────────
# For each race, compute the constructor's average finish position so far that year
df['constructor_avg_pos'] = (
    df.groupby(['year', 'constructorId'])['positionOrder']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# ── Feature 4: Binary — did driver start in the top 3 grid positions? ──────
df['is_top3_grid'] = (df['grid_position'] <= 3).astype(int)

# ── Feature 5: Season (normalised to 0–1 range) ────────────────────────────
df['year_norm'] = (df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min())

# ── Encode constructor as numeric (mean-encode: replace with team's avg podium rate) ──
constructor_podium_rate = df.groupby('name_constructor')['podium'].mean()
df['constructor_podium_rate'] = df['name_constructor'].map(constructor_podium_rate)

# Preview the new features
feature_cols = ['grid_position', 'rolling_pts_3', 'constructor_avg_pos',
                'is_top3_grid', 'year_norm', 'constructor_podium_rate']

print('New features preview:')
df[feature_cols + ['podium']].dropna().head(8)

---
## 6. Prepare Modelling Dataset

We split into features (X) and target (y), then into train and test sets.

**Temporal split strategy:** Instead of a random 80/20 split, we split by year:
- **Train:** 2010–2020
- **Test:** 2021–2023

This is more realistic — in the real world, you train on past races and predict future ones. A random split would let training data 'leak' information from the same season as the test data.

In [ ]:
FEATURES = ['grid_position', 'rolling_pts_3', 'constructor_avg_pos',
            'is_top3_grid', 'year_norm', 'constructor_podium_rate']
TARGET = 'podium'

# Drop rows with any missing feature values
model_df = df[FEATURES + [TARGET, 'year']].dropna().copy()

# Temporal split
train_df = model_df[model_df['year'] <= 2020]
test_df  = model_df[model_df['year'] > 2020]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

print(f'Training set:  {len(X_train):,} rows ({y_train.mean():.1%} podium rate)')
print(f'Test set:      {len(X_test):,} rows ({y_test.mean():.1%} podium rate)')

---
## 7. Train Models

We train three classifiers:

1. **Logistic Regression** — linear model, great baseline. Easy to interpret via coefficients.
2. **Decision Tree** — our main interpretable model. We limit `max_depth=4` to prevent overfitting and keep the tree visualisable.
3. **Random Forest** — ensemble of many trees, generally more accurate but less interpretable. Good for comparison.

The key question: *does more complexity always mean better results?*

In [ ]:
# ── Model definitions ───────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

# ── Train and evaluate all models ──────────────────────────────────────────
results_summary = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Cross-validated accuracy on training set (5-fold)
    cv_acc = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()

    results_summary.append({
        'Model': name,
        'Test Accuracy': f'{acc:.3f}',
        'F1-Score': f'{f1:.3f}',
        'ROC-AUC': f'{auc:.3f}',
        'CV Accuracy (train)': f'{cv_acc:.3f}'
    })

results_df = pd.DataFrame(results_summary)
print('\n=== Model Comparison ===')
print(results_df.to_string(index=False))

---
## 8. Visualise the Decision Tree

This is the centrepiece of an **Interpretable ML** project.

We can literally read the tree's decisions:
- Each node shows the feature and threshold being tested
- The colour shows majority class (orange = podium, blue = no podium)
- The `value` line shows [no_podium_count, podium_count] at that node

Try to trace a path from root to leaf — that's a human-readable rule the model learned.

In [ ]:
dt = models['Decision Tree']

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    dt,
    feature_names=FEATURES,
    class_names=['No Podium', 'Podium'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('Decision Tree — F1 Podium Prediction (max_depth=4)', fontsize=13, pad=14)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

# Also print as text — great for presentations
print('\nText representation of the tree:')
print(export_text(dt, feature_names=FEATURES))

---
## 9. Feature Importance

Both Decision Tree and Random Forest expose `.feature_importances_` — a measure of how much each feature reduces impurity (uncertainty) across all splits.

**What to discuss in your presentation:**
- Is `grid_position` the most important? (It almost certainly will be.)
- How does the ranking differ between the Tree and the Forest?
- Why might `constructor_podium_rate` and `rolling_pts_3` be correlated?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, model) in zip(axes, [('Decision Tree', models['Decision Tree']),
                                      ('Random Forest', models['Random Forest'])]):
    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    importances.plot(kind='barh', ax=ax, color='#5B8DB8', edgecolor='white')
    ax.set_title(f'{name} — feature importance')
    ax.set_xlabel('Importance (Gini impurity reduction)')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Evaluation — Confusion Matrix

A confusion matrix shows the four possible outcomes:

| | Predicted No Podium | Predicted Podium |
|---|---|---|
| **Actual No Podium** | True Negative ✓ | False Positive (model over-confident) |
| **Actual Podium** | False Negative (missed!) | True Positive ✓ |

Given our class imbalance (~15% podium rate), **False Negatives** (missed podiums) are more interesting than raw accuracy to discuss.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Podium', 'Podium'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.suptitle('Confusion matrices (test set)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Evaluation — ROC Curves

The **ROC curve** shows the trade-off between:
- **True Positive Rate** (sensitivity) — how many actual podiums we catch
- **False Positive Rate** — how many non-podiums we wrongly call podiums

**AUC (Area Under the Curve)** summarises this in one number:
- AUC = 0.5 → random guessing
- AUC = 1.0 → perfect model

AUC is more meaningful than accuracy for imbalanced classes.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

colors = {'Logistic Regression': '#5B8DB8', 'Decision Tree': '#E07B54', 'Random Forest': '#6AAB72'}

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colors[name], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves — all models')
ax.legend(fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Error Analysis — What Did the Model Get Wrong?

This is the most interesting part of an Interpretable ML project.

We look at **False Negatives** — races where a driver actually podiumed but the model missed it.

Common patterns to look for:
- Driver started outside top 10 but still podiumed (safety car, rain race, DNFs ahead)
- New team doing unusually well (e.g. early Red Bull rise)
- Driver in poor form who had a one-off great result

These cases teach you what the model *can't* know from structured data alone.

In [ ]:
dt = models['Decision Tree']
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

# Add back the contextual columns for human-readable analysis
test_context = test_df.reset_index(drop=True)[['year', 'name_race', 'surname', 'name_constructor']]

analysis_df = test_context.copy()
analysis_df['actual_podium']    = y_test_reset
analysis_df['predicted_podium'] = dt.predict(X_test_reset)
analysis_df['podium_prob']      = dt.predict_proba(X_test_reset)[:, 1]
analysis_df = pd.concat([analysis_df, X_test_reset], axis=1)

# False Negatives: driver podiumed but model said no
false_negatives = analysis_df[
    (analysis_df['actual_podium'] == 1) & (analysis_df['predicted_podium'] == 0)
].sort_values('podium_prob')

print(f'Total false negatives (missed podiums): {len(false_negatives)}')
print('\nExamples where the model was most wrong (lowest confidence for an actual podium):')
false_negatives[['year', 'name_race', 'surname', 'name_constructor',
                  'grid_position', 'rolling_pts_3', 'podium_prob']].head(10)

---
## 13. Overfitting Investigation — Tree Depth Experiment

As an extra section, we explore how tree depth affects bias vs variance:
- **Shallow tree (depth 1–2):** High bias — too simple, misses patterns
- **Deep tree (depth 10+):** High variance — memorises training data, fails on test data
- **Sweet spot (depth 3–5):** Balances both

This is a core ML concept and a great visual for your presentation.

In [ ]:
depths = range(1, 16)
train_accs, test_accs = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, tree.predict(X_train)))
    test_accs.append(accuracy_score(y_test, tree.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_accs, label='Train accuracy', color='#5B8DB8', linewidth=2, marker='o', markersize=4)
ax.plot(depths, test_accs,  label='Test accuracy',  color='#E07B54', linewidth=2, marker='o', markersize=4)
ax.axvline(x=4, color='gray', linestyle='--', alpha=0.5, label='Our model (depth=4)')
ax.set_xlabel('Tree max depth')
ax.set_ylabel('Accuracy')
ax.set_title('Bias–Variance trade-off: training vs test accuracy by tree depth')
ax.legend()
ax.set_ylim([0.8, 1.01])

# Annotate overfitting region
ax.annotate('Overfitting zone\n(train >> test)', xy=(12, test_accs[11]),
            xytext=(11, 0.88), fontsize=9, color='#E07B54',
            arrowprops=dict(arrowstyle='->', color='#E07B54'))

plt.tight_layout()
plt.savefig('bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 14. Summary

### What we built
A binary classifier to predict F1 podium finishes using Ergast CSV data, focusing on interpretability.

### Key findings
- **Grid position** is the dominant feature — qualifying is highly predictive of race outcome
- **Rolling form** (recent points) adds meaningful signal beyond grid position alone
- **Constructor quality** (team avg position) reflects car performance and proves important
- The **Decision Tree at depth 4** achieves a good balance: interpretable, non-overfitted, competitive with Random Forest on AUC
- **Error analysis** revealed that the model struggles most with chaotic races (rain, safety car, high-grid DNFs) — these require context the structured features don't capture

### Possible extensions
- Add weather data (OpenWeatherMap API)
- Add pit stop data from `pit_stops.csv`
- Try SHAP values for per-prediction explanations
- Build a prediction app for an upcoming race